# 键值型数据库 lmdb
适用于快速持久化存储、高并发读取的 *键值型* 数据库。
- 内存映射
  - LMDB 刚启动时加载到物理内存的只有元数据页、B+ 树的根页、空闲页数据库的根页、少量操作系统预读的页面
  - 当访问之前未访问过的页时，才会从磁盘加载到物理内存
- 单一写，高并发读
  - 任何时候只允许一个 *写事务* 在活动，但可以同时存在无数个 *只读事务*
  - 写事务
    - 不会直接在原页上修改，而是把要修改的页复制一份，在新复制的页面上修改
    - 修改完所有相关页面后，原子性地更新一个目录（元数据页），让目录指向新的修改版本，旧页保持不变
  - 读事务
    - 读取时，始终只会读取开始访问时的版本，即使有写事务也不受影响
- B+ 树结构
  - 实际数据只存储在叶节点中，所以每个内部节点可容纳更多的键/指针，即树的高度很低，搜索效率很高
  - 叶节点之间有双向链接，所以一次定位后，后续可以顺序访问
- 空闲页管理
  - lmdb 维护一个也是由B+树实现的 *空闲页数据库*
  - 每次更新事务会产生旧的页面，对于旧页面，当不再被任何活跃的只读事务引用时，会从空闲页数据库中清理出来，从而避免数据库无限膨胀

## B树 和 B+树
**B树** 是一种多路平衡查找树（泛化的二叉搜索树），**阶** 是指每个非叶节点有最大子节点个数。
- 内部节点：结构为 $[{P_0},({K_1},{D_1}),{P_1},({K_2},{D_2}), \cdots ,({K_n},{D_n}),{P_n}]$
  - $P_i$：是指向子节点的指针
    - $P_i$ 指向的子树中，所有的键都大于 $K_i$ 且小于 $K_{i+1}$
  - $K_i$：是键，*$n$ 个键对应 $n+1$ 个指针*
  - $D_i$：与 $K_i$ 关联的数据
- 叶节点：结构为 $[({K_1},{D_1}),({K_2},{D_2}), \cdots ,({K_n},{D_n})]$

而 **B+树**
- 内部节点：结构为 $[{P_0},K,{P_1},K, \cdots ,K,{P_n}]$
  - 注意这里的 $K$ 是由叶节点的键生成的
    ```
             [ 5, 7 ] 
            /    |    \
           /     |     \
    [(1, D₁)] <-> [ (5, D₅) ] <-> [ (7, D₇), (8, D₈) ]  <-- 叶子节点层
    ```
- 叶节点：$[指向上一个和下一个叶节点的指针,({K_1},{D_1}),({K_2},{D_2}), \cdots ,({K_n},{D_n})]$

## 存储架构
```mermaid
graph TD
    %% 顶部：应用程序视图
    subgraph "A. 应用程序 / 进程空间"
        direction LR
        App["你的应用程序"]
        LMDB_API("LMDB C API (mdb_env, mdb_txn)")
        App --- LMDB_API
    end

    %% 中间层：内存映射区域
    subgraph "B. 内存映射区域"
        direction LR
        MMAP["data.mdb 文件映射到虚拟地址空间"]
        MMAP_PAGES["逻辑页框 (Page Frames)"]
        MMAP --- MMAP_PAGES
    end
    LMDB_API --- MMAP

    %% 物理存储层：磁盘文件
    subgraph "C. 物理磁盘存储"
        direction LR
        DATA_MDB["data.mdb (主数据文件)"]
        LOCK_MDB["lock.mdb (锁文件/元信息)"]
    end
    MMAP_PAGES -- "OS Page Cache / 虚拟内存管理" --> DATA_MDB

    %% data.mdb 内部结构 (B+树)
    subgraph "D. data.mdb 内部结构 (B+树组织)"
        direction TB

        META_PAGES_GROUP[元数据页管理]
        subgraph "元数据页 (Meta Pages)"
            META0["Meta Page 0"]
            META1["Meta Page 1"]
        end
        META_DETAIL(最新TxnID, Root Ptr, FreeDB Ptr)
        META0 -- "包含" --> META_DETAIL
        META1 -- "包含" --> META_DETAIL
        
        BPLUS_TREE_GROUP[B+树索引和数据]
        subgraph "B+树结构 (Root 到 Leaf)"
            ROOT_PAGE["根页 (Root Page, P_BRANCH)"]
            BRANCH_PAGES_GROUP(分支页层)
            subgraph "分支页 (Branch Pages, P_BRANCH)"
                BRANCH_PAGE1["[PtrA, Key1, PtrB, Key2, PtrC]"]
                BRANCH_PAGE2["[PtrD, Key3, PtrE]"]
            end
            LEAF_PAGES_GROUP(叶子页层)
            subgraph "叶子页 (Leaf Pages, P_LEAF)"
                LEAF_PAGE1["[(K1, D1), (K2, D2)]"]
                LEAF_PAGE2["[(K3, D3), (K4, D4)]"]
                LEAF_PAGE3["[(K5, D5)]"]
            end
        end

        OVERFLOW_PAGES_GROUP[溢出页]
        subgraph "溢出页 (Overflow Pages, P_OVERFLOW)"
            OVERFLOW_PAGE["存储大Value数据"]
        end
        LEAF_PAGE1 -- "包含指向" --> OVERFLOW_PAGE

        FREE_DB_GROUP[空闲页管理]
        subgraph "空闲页数据库 (FreeDB)"
            FREE_DB_ROOT["FreeDB Root Page"]
            FREE_LISTS(TxnID -> List of Freed Pages)
        end
        FREE_DB_ROOT -- "管理" --> FREE_LISTS
    end
    
    %% lock.mdb 内部结构
    subgraph "E. lock.mdb 内部结构 (协调与锁)"
        direction TB
        WRITE_MUTEX["写互斥锁"]
        READER_TABLE["读者列表 (Reader Table)"]
        LATEST_TXNID["最新提交事务ID"]
    end

    %% 关键连接和流
    LMDB_API -- "读写操作" --> BPLUS_TREE_GROUP
    LMDB_API -- "事务管理" --> E
    LMDB_API -- "环境配置" --> META_PAGES_GROUP

    META_DETAIL -- "指向 B+树根" --> ROOT_PAGE
    META_DETAIL -- "指向 FreeDB" --> FREE_DB_ROOT
    
    ROOT_PAGE --> BRANCH_PAGE1
    ROOT_PAGE --> BRANCH_PAGE2
    BRANCH_PAGE1 --> LEAF_PAGE1
    BRANCH_PAGE1 --> LEAF_PAGE2
    BRANCH_PAGE2 --> LEAF_PAGE3

    LEAF_PAGE1 -- "叶子链表(前驱)" --> LEAF_PAGE2
    LEAF_PAGE2 -- "叶子链表(后继)" --> LEAF_PAGE1
    LEAF_PAGE2 -- "叶子链表(前驱)" --> LEAF_PAGE3
    LEAF_PAGE3 -- "叶子链表(后继)" --> LEAF_PAGE2

    LATEST_TXNID -- "被读事务更新" --> READER_TABLE
    WRITE_MUTEX -- "由写事务获取" --> DATA_MDB
    FREE_DB_ROOT -- "回收利用页面" --> BPLUS_TREE_GROUP

    %% 样式定义
    classDef main_box fill:#e0f7fa,stroke:#0097a7,stroke-width:2px
    class App,LMDB_API,MMAP,MMAP_PAGES,DATA_MDB,LOCK_MDB,META_PAGES_GROUP,BPLUS_TREE_GROUP,OVERFLOW_PAGES_GROUP,FREE_DB_GROUP,WRITE_MUTEX,READER_TABLE,LATEST_TXNID main_box
    classDef sub_box fill:#fff8e1,stroke:#ffa000,stroke-width:1.5px
    class META0,META1,ROOT_PAGE,BRANCH_PAGE1,BRANCH_PAGE2,LEAF_PAGE1,LEAF_PAGE2,LEAF_PAGE3,OVERFLOW_PAGE,FREE_DB_ROOT sub_box
    classDef detail_box fill:#e8f5e9,stroke:#4caf50,stroke-width:1px
    class META_DETAIL,FREE_LISTS detail_box
    classDef arrow_label fill:#f8bbd0,stroke:#e91e63,stroke-width:0px,color:#e91e63
```

## 使用架构
```mermaid
graph TD
    subgraph "A. 初始化与环境 (Initialization & Environment)"
        A["创建 MDB_env 句柄<br/>mdb_env_create()"] --> B{"配置环境<br/>mdb_env_set_mapsize()<br/>mdb_env_set_maxdbs()"}
        B --> C["打开环境<br/>mdb_env_open()<br/>在指定目录创建 data.mdb 和 lock.mdb"]
    end

    subgraph "B. 事务管理 (Transaction Management)"
        C --> D{"启动事务<br/>mdb_txn_begin()"}
        D -- "读/写模式" --> Txn_RW["读写事务 (Write Txn)"]
        D -- "只读模式 (MDB_RDONLY)" --> Txn_R["只读事务 (Read Txn)"]
    end

    subgraph "C. 数据库操作 (Database Operations)"
        Txn_RW --> E_RW{"打开数据库句柄 (DBI)<br/>mdb_dbi_open()"}
        Txn_R --> E_R{"打开数据库句柄 (DBI)<br/>mdb_dbi_open()"}

        E_RW --> F_RW["写操作 (CRUD)"]
        E_R --> F_R["读操作 (Read-Only)"]

        subgraph "写操作 (CRUD)"
            direction LR
            PUT["直接写入/更新<br/>mdb_put()"]
            DEL["直接删除<br/>mdb_del()"]
            CURSOR_RW["使用游标进行<br/>增/删/改/查<br/>mdb_cursor_put()<br/>mdb_cursor_del()"]
        end
        F_RW --> PUT
        F_RW --> DEL
        F_RW --> CURSOR_RW
        
        subgraph "读操作 (Read-Only)"
            direction LR
            GET["直接读取<br/>mdb_get()"]
            CURSOR_R["使用游标进行<br/>遍历/查找<br/>mdb_cursor_get()"]
        end
        F_R --> GET
        F_R --> CURSOR_R
    end

    subgraph "D. 结束事务 (Transaction Finalization)"
        Txn_RW --> G_Commit["提交事务<br/>mdb_txn_commit()<br/>(持久化更改)"]
        Txn_RW --> H_Abort["中止事务<br/>mdb_txn_abort()<br/>(回滚更改)"]
        Txn_R --> I_Abort["结束事务<br/>mdb_txn_abort()<br/>或 mdb_txn_reset()"]
    end

    subgraph "E. 清理 (Cleanup)"
        J["关闭环境<br/>mdb_env_close()<br/>释放所有资源"]
    end
    G_Commit --> J
    H_Abort --> J
    I_Abort --> J

    %% 样式
    style C fill:#d4edda,stroke:#155724
    style D fill:#cce5ff,stroke:#004085
    style G_Commit fill:#d4edda,stroke:#155724
    style H_Abort fill:#f8d7da,stroke:#721c24
    style I_Abort fill:#fff3cd,stroke:#856404
```

## LMDB数据库接口 WtLMDB.hpp

# 堆栈跟踪 StackTracer

## 操作系统信号回调 SignalHook.hpp

捕获操作系统发出的各种信号，并自定义回调函数进行处理
- 例如在终端按下 `Ctrl+C` 时，系统会发送一个 `SIGINT` (中断) 信号
- 当程序访问非法内存时，会收到 `SIGSEGV` (段错误) 信号

自定义日志回调函数和退出回调函数，然后使用下述函数注册：
```cpp
/* 安装信号处理钩子
 * @param cbLog 信号日志回调函数，用于输出信号处理信息
 * @param sigHandler 可选的自定义退出处理函数，NULL表示使用默认处理
 */
void install_signal_hooks(TracerLogCallback cbLog, ExitHandler sigHandler = NULL)
{
	g_cbSignalLog = cbLog;                    // 设置全局日志回调函数
	g_exitHandler = sigHandler;               // 设置全局退出处理函数
	for (int s = 1; s < NSIG; s++)            // 遍历所有信号编号
	{
		signal(s, handle_signal);             // 为每个信号安装处理函数
	}
}
```
- 实际操作系统发送信号后 ——> `handle_signal(int signum)`

# YAML 格式文件管理 yamlcpp

## YAML/JSON 配置文件加载器 WTSCfgLoader.h/cpp

# 压缩解压工具 zstdlib

## 数据压缩解压 WTSCmpHelper.hpp